# Data modeling and warehousing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/data-modeling.ipynb)

Data modeling is where most analytics problems are actually decided: a warehouse can be fast and still report the wrong revenue if the grain, keys, or history handling are wrong. The first half covers the modeling vocabulary interviewers expect (grain, facts, dimensions, star and snowflake schemas, surrogate keys, conformed dimensions, SCD types, bridges, metric contracts) and the Kimball, Inmon, and Data Vault approaches. The second half covers the warehouse built around the model: layers, medallion and lakehouse, staging and incremental loads, point-in-time joins, late-arriving data, physical design, ROLLUP and GROUPING SETS, testing, and a clickstream sessionization case. Every SQL example runs in-process on DuckDB against one shared in-memory database, so the notebook runs top to bottom as written.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is data modeling in analytics and why does it matter?

Data modeling structures data so business questions can be answered reliably and cheaply. An OLTP schema is modeled for correct *writes*: 3NF stores each fact once so an update cannot create a contradiction. An analytics model is modeled for correct *reads*: a fact table at a declared grain plus dimensions that describe it. The trade is write complexity for read simplicity.

The model lives in layers (raw, cleaned and conformed, then marts; see the warehouse layers section later), and three decisions recur at every layer:

1. **Grain**: what does one row represent?
2. **Keys**: natural, surrogate, or composite, and how they join.
3. **Change handling**: overwrite (SCD1), version history (SCD2), or append-only events.

Normalization levels to know: 1NF makes columns atomic, 2NF removes partial-key dependencies, 3NF removes transitive dependencies. Dimensional modeling deliberately denormalizes dimensions so a report needs one join per dimension.

Interview signal: a candidate who starts with "what is the grain and what does the business call a customer?" shows design judgment; one who starts writing `JOIN`s shows only SQL.

Common mistake: copying source tables 1:1 into the warehouse and calling it a model. A 1:1 copy inherits the source grain, so every consumer re-derives the same metric and gets a different answer.

## What are conceptual, logical, and physical models?

| Model | Shows | Keys | Audience |
|---|---|---|---|
| Conceptual | main entities and that they relate | none | business |
| Logical | entities, all attributes, relationships, many-to-many resolved, normalized | primary and foreign keys | architects |
| Physical | tables, columns, data types, indexes, partitions, denormalization; specific to one engine | enforced or declared constraints | engineers |

ER modeling normalizes for OLTP writes; dimensional modeling deliberately denormalizes dimensions for reads. A fact table is still close to normalized: it holds foreign keys plus measures that depend on the full key.

## What is grain and why is it the first question to answer?

Grain is the level of detail of a table, such as "one row per order line" or "one row per account per day". It is a contract with three parts: the entity, the event or period, and the uniqueness rule. Grain and primary key are the same statement written twice; if they disagree, the model is wrong.

Kimball's four-step design process puts the grain second, before any column exists:

1. Choose the business process (retail sales).
2. Declare the grain: the sentence "one row per ___" (one row per order line).
3. Identify the dimensions that describe that grain (date, product, customer).
4. Identify the numeric facts that are true at that grain (quantity, net amount).

Grain fixes how each measure behaves:

+ **Additive** across all dimensions: quantity, amount.
+ **Semi-additive**, usually not across time: account balance, inventory on hand. Use `avg`, `min`, `max`, or last value over time, never `sum`.
+ **Non-additive**: ratios and percentages. Store numerator and denominator; compute the ratio at query time.

Choose the **lowest grain that answers the questions** and that you can afford to store: you can always aggregate up, never split a row back apart. "Revenue by product by month and average order value" needs order-line grain, because AOV is `sum(line_amount) / count(distinct order_id)`.

Common grain mistakes:

+ Storing order-level values (shipping, tax) on every order-line row, then summing them. Keep them in a `fact_order` at order grain, or allocate them to lines with a documented rule, never both.
+ Mixing snapshot and event grain in one table so a `sum` silently multiplies.
+ No declared key, so nothing prevents duplicate loads.
+ Choosing the grain of today's dashboard instead of the business process.

The cell below builds a small star schema in DuckDB that the rest of the notebook reuses, and encodes the grain as a uniqueness test.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
create table dim_date as
select cast(strftime(d, '%Y%m%d') as int) as date_sk,
       cast(d as date)                    as date_value,
       strftime(d, '%Y-%m')               as year_month,
       dayofweek(d) in (0, 6)             as is_weekend
from range(date '2026-09-01', date '2026-11-01', interval 1 day) t(d);

create table dim_product (
  product_sk bigint primary key, product_id varchar, product_name varchar,
  category varchar, department varchar
);
insert into dim_product values
  (1, 'P1', 'Trail shoe', 'Footwear', 'Outdoor'),
  (2, 'P2', 'Rain jacket', 'Apparel', 'Outdoor'),
  (3, 'P3', 'Desk lamp', 'Lighting', 'Home');

create table fact_order_line (
  order_id varchar not null,          -- degenerate dimension
  line_number int not null,
  customer_id varchar not null,
  product_sk bigint not null,
  order_date_sk int not null,
  order_ts timestamp not null,
  quantity int not null,
  net_amount decimal(14, 2) not null,
  unique (order_id, line_number)      -- grain guard
);
insert into fact_order_line values
  ('O1', 1, 'C1', 1, 20260920, '2026-09-20 10:00', 1, 120.00),
  ('O1', 2, 'C1', 3, 20260920, '2026-09-20 10:00', 2,  60.00),
  ('O2', 1, 'C2', 2, 20261003, '2026-10-03 09:30', 1, 200.00),
  ('O3', 1, 'C1', 1, 20261015, '2026-10-15 18:45', 2, 240.00),
  ('O3', 2, 'C1', 2, 20261015, '2026-10-15 18:45', 1, 180.00);
""")

# grain test: the declared key must be unique (query must return zero rows)
dupes = con.execute("""
    select order_id, line_number, count(*) from fact_order_line
    group by all having count(*) > 1
""").fetchall()
assert dupes == []
print("fact_order_line rows:", con.execute("select count(*) from fact_order_line").fetchone()[0])

### Additivity in practice

The next cell makes the additivity rules concrete. Summing a daily inventory snapshot across days invents stock that never existed, and averaging monthly average order values weights a one-order month the same as a two-order month. The fix in both cases is the rule from the grain section: take the period-end value for semi-additive measures, and recompute ratios from summed components.

In [ ]:
con.execute("""
-- periodic snapshot: one row per product per day; units_on_hand is semi-additive
create table fact_inventory_daily (
  snapshot_date_sk int not null, product_sk bigint not null, units_on_hand int not null,
  primary key (snapshot_date_sk, product_sk)
);
insert into fact_inventory_daily values
  (20261029, 1, 10), (20261029, 2, 4),
  (20261030, 1,  8), (20261030, 2, 4),
  (20261031, 1,  7), (20261031, 2, 3);
""")

summed_over_time = con.execute("select sum(units_on_hand) from fact_inventory_daily").fetchone()[0]
month_end_stock = con.execute("""
    select sum(units_on_hand) from fact_inventory_daily
    where snapshot_date_sk = (select max(snapshot_date_sk) from fact_inventory_daily)
""").fetchone()[0]
print("summed across days:", summed_over_time, "| month-end stock:", month_end_stock)
assert (summed_over_time, month_end_stock) == (36, 10)   # 36 units never existed

# non-additive: recompute average order value from components, never average the averages
avg_of_monthly_aov, true_aov = (float(v) for v in con.execute("""
    with monthly as (
        select d.year_month, sum(f.net_amount) as revenue, count(distinct f.order_id) as orders
        from fact_order_line f
        join dim_date d on d.date_sk = f.order_date_sk
        group by all
    )
    select round(avg(revenue / orders), 2), round(sum(revenue) / sum(orders), 2)
    from monthly
""").fetchone())
print("average of monthly AOV:", avg_of_monthly_aov, "| true AOV:", true_aov)
assert (avg_of_monthly_aov, true_aov) == (245.0, 266.67)

## What is the difference between fact and dimension tables?

Facts store measurable events or snapshots (orders, payments, balances) plus foreign keys to dimensions: tall, narrow, growing with events, usually partitioned by date. Dimensions store descriptive attributes (customer, product, date) used to filter and group: short, wide, growing with entities. Facts answer "what happened, how much"; dimensions answer "who, what, where, when".

Practical rules:

+ Fact foreign keys reference the dimension's surrogate key so every join is many-to-one and no fact row duplicates.
+ Not every number is a measure: tax rate, discount percentage, and latitude are descriptive and belong in a dimension.
+ Descriptive text in the fact (to "save a join") repeats per event, turning a rename into a mass update. The exception is a **degenerate dimension** such as `order_id`: an identifier at the fact grain with no attributes of its own, kept in the fact with no dimension table.
+ A **junk dimension** is the opposite move: many low-cardinality flags collapsed into one small table so the fact does not grow a column per flag.

## What is a star schema and what are its benefits?

A star schema is a central fact table joined to flat, denormalized dimensions on single-column surrogate keys. Every dimension join is many-to-one, so the fact row count never changes and measures stay additive. BI tools like stars because they can infer the join graph and generate `GROUP BY` without fan-out.

```text
          dim_date
             |
dim_product -- fact_order_line -- dim_customer
             |
          dim_store
```

The star contract is one row per dimension key. A dimension with one row per address (or per SCD2 version, joined without a filter) duplicates every fact row. Adding a second fact to the diagram does not make it a star: joining two facts at different grains fans out (see double counting below).

In [ ]:
star = con.execute("""
    select d.year_month,
           p.category,
           sum(f.net_amount) as net_revenue,
           sum(f.quantity)   as units
    from fact_order_line f
    join dim_date    d on d.date_sk = f.order_date_sk
    join dim_product p on p.product_sk = f.product_sk
    group by all
    order by all
""").fetchall()
for row in star:
    print(row)

# a many-to-one star join never changes the measure total
star_total = sum(row[2] for row in star)
fact_total = con.execute("select sum(net_amount) from fact_order_line").fetchone()[0]
assert star_total == fact_total == 800

## Star schema vs snowflake schema: what are the trade-offs?

A snowflake normalizes dimensions into sub-tables (`dim_product -> dim_category -> dim_department`). It reduces redundancy and makes a shared sub-attribute single-sourced, at the cost of more joins.

```sql
-- snowflaked
create table dim_product  (product_sk bigint primary key, product_name varchar, category_sk bigint);
create table dim_category (category_sk bigint primary key, category_name varchar, department_sk bigint);
-- star: the same attributes flattened
create table dim_product  (product_sk bigint primary key, product_name varchar,
                           category_name varchar, department_name varchar);
```

| Aspect | Star | Snowflake |
|---|---|---|
| Joins per report | 1 hop per dimension | multiple hops |
| Storage | more redundancy (compresses well) | less redundancy |
| Ease for BI users | high | lower |
| Dimension maintenance | rewrite a wide row | update a small table |
| Best fit | BI marts, self-service | huge or widely shared sub-dimensions |

On a columnar MPP engine the argument tilts further toward stars: repeated text compresses well, so the storage saving is small, and small dimensions are broadcast to every node so a one-hop join moves no fact rows, while a chain of joins adds stages and chances for a shuffle.

Interview answer: use a star in the consumption layer. Snowflake only when a sub-attribute is genuinely shared by many dimensions (global geography, cost-center hierarchy) or the dimension is enormous. If you keep a snowflake, publish flattened views generated from it, not hand-maintained copies that drift.

Related terms: a **galaxy schema** (fact constellation) is several fact tables sharing dimensions; a **cube** is the logical multidimensional view of a star (dimension members on the edges, measures in the cells).

The cell below snowflakes `dim_product` into category and department tables and proves both shapes return the same rows, with one more join per hierarchy level.

In [ ]:
con.execute("""
create table dim_department as
select row_number() over (order by department) as department_sk, department as department_name
from (select distinct department from dim_product);

create table dim_category as
select row_number() over (order by p.category) as category_sk, p.category as category_name, d.department_sk
from (select distinct category, department from dim_product) p
join dim_department d on d.department_name = p.department;

create table dim_product_sf as
select p.product_sk, p.product_id, p.product_name, c.category_sk
from dim_product p
join dim_category c on c.category_name = p.category;
""")

snowflake = con.execute("""
    select d.year_month,
           c.category_name   as category,
           sum(f.net_amount) as net_revenue,
           sum(f.quantity)   as units
    from fact_order_line f
    join dim_date       d on d.date_sk = f.order_date_sk
    join dim_product_sf p on p.product_sk = f.product_sk     -- product -> category is an extra hop
    join dim_category   c on c.category_sk = p.category_sk
    group by all
    order by all
""").fetchall()
print("star rows == snowflake rows:", snowflake == star)
assert snowflake == star

## What are the main types of fact tables?

| Type | Grain | Behavior | Answers |
|---|---|---|---|
| Transaction | one row per event | insert-only, sparse | "how many / how much?" |
| Periodic snapshot | one row per entity per period | dense, usually semi-additive | "what was the balance / how many open on the 15th?" |
| Accumulating snapshot | one row per process instance | updated in place as milestones complete; lag columns | "how long did each stage take?" |
| Factless | one row per event or relationship, no measure | counted, not summed | coverage, eligibility, attendance |

```sql
-- periodic snapshot: one row per account per day
create table fact_account_balance_daily (
  account_sk bigint, snapshot_date_sk int, closing_balance numeric(18, 2),
  primary key (account_sk, snapshot_date_sk)
);
-- accumulating snapshot: one row per order, milestones updated in place
create table fact_order_fulfillment (
  order_sk bigint primary key,
  ordered_date_sk int, packed_date_sk int, shipped_date_sk int, delivered_date_sk int,
  days_to_ship int
);
```

Transaction and snapshot facts cannot substitute for each other. Modeling a state question ("open tickets") as a transaction fact forces you to net deltas by hand; model it as a periodic snapshot and the metric is a row count. Accumulating snapshots make stage durations cheap (`avg(days_in_transit)`) at the cost of in-place updates that must be idempotent.

The fact type also decides what late data means: a late transaction is appended to its partition, a late snapshot restates a period, and a late milestone updates an accumulating row.

## What is a surrogate key and when should you use it?

A natural key identifies a thing in the source; a surrogate key identifies a row in your dimension, so the warehouse controls identity and versioning. Use one when source keys are unstable or recycled, when multiple sources have overlapping key spaces, and whenever you keep SCD2 history.

+ Type 1 keeps one surrogate key per entity forever; Type 2 issues a new surrogate key per version while the natural key repeats.
+ Facts store the surrogate key resolved at load time, which makes point-in-time joins exact.
+ A deterministic hash of the business key is parallel-safe (every worker computes the same value, no coordination). An identity/sequence is compact but needs a central generator.
+ Keep keys meaningless; never encode business meaning in digits.
+ Assign them in one place: two pipelines generating keys independently split one customer's history in two.
+ Hash the **business key only**, including the source system when key spaces overlap. If the hash includes a tracked attribute like `segment`, a segment change mints a new entity identity and cross-version reporting breaks.

In [ ]:
keys = con.execute("""
    select source_system, customer_id,
           md5(source_system || '|' || customer_id) as customer_hk
    from (values ('crm', '1001'), ('billing', '1001'), ('crm', '1001')) t(source_system, customer_id)
""").fetchall()
for row in keys:
    print(row)

# deterministic: same input -> same key; source system prevents cross-system collisions
assert keys[0][2] == keys[2][2]
assert keys[0][2] != keys[1][2]

## What is a conformed dimension?

A conformed dimension is shared across facts or marts with the same table, the same surrogate keys, and the same attribute definitions, for example one `dim_customer` used by `fact_order` and `fact_support_ticket`. That sameness makes **drill-across** valid: aggregate each fact to a shared dimension attribute at its own grain, then join the small results. Never join fact to fact directly. A **conformed fact** is the same idea for measures: the same name, grain, and business rule wherever it appears.

Keeping it conformed: one owner, one refresh schedule, one definition per attribute, and a **bus matrix** recording which processes use which conformed dimensions. `dim_date` is the most common conformed dimension.

```text
                 dim_date  dim_customer  dim_product  dim_store
fact_order_line     X          X             X            X
fact_payment        X          X                          X
fact_inventory      X                        X            X
```

Gotcha: two marts that each build their own `dim_customer` with different segment logic are not conformed even when column names match; so is conforming the keys but not the attributes. The drift stays silent until a cross-domain report disagrees. The classic case is `dim_date_sales` and `dim_date_finance` with different fiscal calendars and no mapping.

The cell below drills across sales and returns on the conformed product dimension, then shows what joining the two facts row to row does to both totals.

In [ ]:
con.execute("""
create table fact_return (
  return_id varchar primary key, return_date_sk int not null, product_sk bigint not null,
  refund_amount decimal(14, 2) not null
);
insert into fact_return values ('R1', 20261005, 2, 50.00), ('R2', 20261020, 1, 120.00);
""")

# drill across: aggregate each fact to the shared grain, then join the small results
drill = con.execute("""
    with sales as (
        select p.category, sum(f.net_amount) as revenue
        from fact_order_line f join dim_product p using (product_sk) group by all
    ), returns as (
        select p.category, sum(r.refund_amount) as refunds
        from fact_return r join dim_product p using (product_sk) group by all
    )
    select category, revenue, coalesce(refunds, 0) as refunds, revenue - coalesce(refunds, 0) as net
    from sales left join returns using (category)
    order by category
""").fetchall()
for row in drill:
    print(row)
assert drill == [("Apparel", 380, 50, 330), ("Footwear", 360, 120, 240), ("Lighting", 60, 0, 60)]

# fact-to-fact join: refunds repeat once per order line and Lighting (no returns) disappears
naive_revenue, naive_refunds = con.execute("""
    select sum(f.net_amount), sum(r.refund_amount)
    from fact_order_line f join fact_return r using (product_sk)
""").fetchone()
print("fact-to-fact join: revenue", naive_revenue, "| refunds", naive_refunds)
assert (naive_revenue, naive_refunds) == (740, 340)   # true values: 800 and 170

## What is SCD (Slowly Changing Dimension)?

SCD describes how a dimension handles attribute changes over time. Pick the type **per attribute**, not per table: a name typo is Type 1, a segment change that should keep historical revenue attributed to the old segment is Type 2. A dimension that mixes types this way is called a **hybrid**.

| Type | Behavior | History | Typical use |
|---|---|---|---|
| 0 | never changes | frozen | immutable codes, original signup date |
| 1 | overwrite in place | none | corrections, phone numbers |
| 2 | new row + validity range | full | segment, address, plan, ownership |
| 3 | previous value in an extra column | one step | simple before/after comparisons |
| 4 | separate history table | full, outside main dim | huge or audit-heavy dimensions |
| 6 | Type 2 + current value on every row | full + current | report by current and historical value |

Type 2 columns: new surrogate key per version, natural key, `valid_from`, `valid_to`, `is_current`, often `version`. Invariants:

+ Ranges are **half-open** `[valid_from, valid_to)`, contiguous, and non-overlapping.
+ Exactly one row per natural key has `is_current = true`.
+ The open row ends at a sentinel such as `9999-12-31`, not `NULL`, so range predicates work without `coalesce`.

Type 1 is cheap but destroys as-of reporting irreversibly; Type 3 answers "what was it before?" but not "what was it in March?". Type 6's current-value column must be rewritten on every version in the same transaction (write amplification). Applying Type 2 to everything makes every typo a version and bloats history nobody queries. When the business cannot state a requirement, default to Type 2 for analytical attributes and Type 1 for corrections.

```sql
-- Type 1: overwrite
update dim_customer as tgt set email = src.email
from stg_customer src where src.customer_id = tgt.customer_id and tgt.is_current;

-- Type 3: keep one previous value
update dim_customer as tgt
set previous_segment = tgt.segment, segment = src.segment, segment_changed_at = current_timestamp
from stg_customer src
where src.customer_id = tgt.customer_id and tgt.is_current
  and tgt.segment is distinct from src.segment;
```

## How do you implement SCD Type 2?

The load is two operations that must be atomic (one transaction or one `MERGE`), or readers can see zero or two current rows:

1. Stage the incoming snapshot and deduplicate to one row per natural key (latest wins).
2. Compare tracked attributes to the current row with a **null-safe** operator (`is distinct from`, or `<=>` in MySQL/Spark) so a `NULL -> value` change is detected.
3. Close changed current rows: `valid_to = run_ts`, `is_current = false`.
4. Insert a new version for every changed or brand-new key with `valid_from = run_ts`, `valid_to = '9999-12-31'`.

Idempotency: take `run_ts` from the batch (the snapshot's effective time), not from `now()` per row, and make change detection compare against the current row, so rerunning the same snapshot inserts nothing. The cell below implements this in DuckDB and proves the rerun is a no-op. On Snowflake, Databricks, or BigQuery the same logic is usually a single `MERGE` that unions changed rows in twice (once to close, once to insert).

Production loads also declare `unique (customer_id, valid_from)` and `check (valid_to > valid_from)`, and use a single surrogate-key generator.

In [ ]:
con.execute("""
create table dim_customer (
  customer_sk bigint primary key, customer_id varchar not null, segment varchar, email varchar,
  valid_from timestamp not null, valid_to timestamp not null, is_current boolean not null
);
insert into dim_customer values
  (1001, 'C1', 'standard', 'c1@x.com', '2026-01-01', '9999-12-31', true),
  (1002, 'C2', 'standard', null,       '2026-01-01', '9999-12-31', true);
create sequence customer_sk_seq start 1003;
create table stg_customer (customer_id varchar, segment varchar, email varchar, updated_at timestamp);
""")

TRACKED_CHANGED = "(d.segment is distinct from s.segment or d.email is distinct from s.email)"


def load_scd2(snapshot, run_ts):
    con.execute("delete from stg_customer")
    con.executemany("insert into stg_customer values (?, ?, ?, ?)", snapshot)
    con.execute("begin")
    con.execute("""
        create or replace temp table src as
        select * from stg_customer
        qualify row_number() over (partition by customer_id order by updated_at desc) = 1
    """)
    con.execute(f"""
        update dim_customer d set valid_to = ?, is_current = false
        from src s
        where d.customer_id = s.customer_id and d.is_current and {TRACKED_CHANGED}
    """, [run_ts])
    con.execute("""
        insert into dim_customer
        select nextval('customer_sk_seq'), s.customer_id, s.segment, s.email, ?, '9999-12-31', true
        from src s
        where not exists (select 1 from dim_customer d where d.customer_id = s.customer_id and d.is_current)
    """, [run_ts])
    con.execute("commit")


snapshot = [
    ("C1", "enterprise", "c1@x.com", "2026-10-01 08:00"),   # tracked change -> new version
    ("C2", "standard", "c2@x.com", "2026-10-01 08:00"),     # NULL -> value is a change too
    ("C3", "standard", "c3@x.com", "2026-10-01 08:00"),     # brand-new customer
]
load_scd2(snapshot, "2026-10-01")
rows_after_first = con.execute("select count(*) from dim_customer").fetchone()[0]
load_scd2(snapshot, "2026-10-02")                             # rerun: must be a no-op
rows_after_rerun = con.execute("select count(*) from dim_customer").fetchone()[0]

for row in con.execute("select * from dim_customer order by customer_id, valid_from").fetchall():
    print(row)
assert rows_after_first == rows_after_rerun == 5
assert con.execute("""
    select count(*) from (select customer_id from dim_customer
                          group by customer_id having count(*) filter (where is_current) <> 1)
""").fetchone()[0] == 0

## How do SCD Type 3 and Type 6 work?

Type 3 adds a `previous_segment` column (the `update` shown in the SCD types section): cheap, but only one prior value survives, so it answers "what was it before?" and never "what was it in March?". Type 6 (1 + 2 + 3) keeps full Type 2 history and adds a `segment_current` column overwritten on every version of the natural key, so one table answers both "revenue by the segment at the time" and "revenue by the segment customers are in now".

The cell below derives `segment_current` with a self-join into a separate `dim_customer_t6`, which leaves the base dimension unchanged for the rest of the notebook. In a production Type 6 dimension the SCD load rewrites that column on every version, in the same transaction that inserts the new version: that is the write amplification mentioned above. The point-in-time section later uses this table to report revenue both ways.

In [ ]:
con.execute("""
create table dim_customer_t6 as
select d.*, c.segment as segment_current
from dim_customer d
join dim_customer c on c.customer_id = d.customer_id and c.is_current;
""")

c1_versions = con.execute("""
    select customer_sk, segment, segment_current, is_current
    from dim_customer_t6 where customer_id = 'C1' order by valid_from
""").fetchall()
for row in c1_versions:
    print(row)

# every version carries the current value; the historical value stays on its own row
assert [(segment, current) for _, segment, current, _ in c1_versions] == [
    ("standard", "enterprise"),
    ("enterprise", "enterprise"),
]

## How do you prevent double counting in analytical models?

Fan-out happens whenever a join is not many-to-one. The usual causes:

+ A dimension with duplicate keys (one row per address, or SCD2 history joined without a current or as-of filter). Referential integrity does not imply uniqueness.
+ A many-to-many relationship joined directly instead of through a bridge.
+ Two facts at different grains joined on a shared key (the **chasm trap**). Aggregate each to a common grain first.

Other guards: semi-additive measures use last value or average over time, never `sum`; `count(distinct key)` can hide a fan-out that `sum` exposes, so test measures, not just counts. Detection is mechanical: compare row count and measure total before and after each join.

In [ ]:
con.execute("""
create table dim_customer_address (customer_id varchar, address_id int, is_primary boolean);
insert into dim_customer_address values ('C1', 1, true), ('C1', 2, false), ('C2', 3, true);
""")

wrong = con.execute("""
    select sum(f.net_amount) from fact_order_line f
    join dim_customer_address a on a.customer_id = f.customer_id
""").fetchone()[0]
right = con.execute("""
    select sum(f.net_amount) from fact_order_line f
    join dim_customer_address a on a.customer_id = f.customer_id and a.is_primary
""").fetchone()[0]
print("fanned-out total:", wrong, "| correct total:", right)

# C1's lines (600) are counted twice because C1 has two address rows
assert right == 800
assert wrong == 800 + 600

## What is a bridge table and how do you model many-to-many relationships?

A bridge resolves a many-to-many relationship (orders with several promotions, accounts with several owners). Its grain is one row per relationship, keyed by the pair of surrogate keys. Two different metrics come out of it, and mixing them is the usual error:

+ **Weighted attribution**: `sum(measure * allocation_weight)`. Reconciles to the parent total only if each parent's weights sum to exactly 1, so test that. With no business rule, use `1 / count` of the parent's relationships.
+ **Influence**: `count(distinct parent_key)`, answering "how many orders were touched". It does not reconcile to revenue.

An unweighted join of the bridge to the fact counts revenue once per relationship. A "primary" flag lets simple reports pick one relationship. Name metrics by their formula so two dashboards don't both show "promotion revenue" computed differently.

In [ ]:
con.execute("""
create table fact_order as
select order_id, sum(net_amount) as revenue from fact_order_line group by order_id;
create table bridge_order_promotion (order_id varchar, promotion varchar, allocation_weight decimal(5, 4));
insert into bridge_order_promotion values
  ('O1', 'BACK2SCHOOL', 0.5), ('O1', 'LOYALTY', 0.5), ('O3', 'LOYALTY', 1.0);
""")

weighted = dict(con.execute("""
    select b.promotion, sum(f.revenue * b.allocation_weight)
    from fact_order f join bridge_order_promotion b using (order_id) group by all
""").fetchall())
unweighted_total = con.execute("""
    select sum(f.revenue) from fact_order f join bridge_order_promotion b using (order_id)
""").fetchone()[0]
influenced = con.execute("select count(distinct order_id) from bridge_order_promotion").fetchone()[0]
bad_weights = con.execute("""
    select order_id from bridge_order_promotion group by order_id having sum(allocation_weight) <> 1
""").fetchall()
print("weighted:", weighted, "| unweighted sum:", unweighted_total, "| influenced orders:", influenced)

promoted_revenue = con.execute("""
    select sum(revenue) from fact_order where order_id in (select order_id from bridge_order_promotion)
""").fetchone()[0]
assert sum(weighted.values()) == promoted_revenue == 600   # reconciles
assert unweighted_total == 780                               # O1 counted twice
assert influenced == 2 and bad_weights == []

## What is a factless fact table?

A factless fact records an event or relationship with no numeric measure. Aggregate it with `count(distinct ...)`, never by summing a placeholder `1` after a join (that double counts).

+ **Event factless**: a login, a class attendance.
+ **Coverage factless**: a relationship that was *possible*, such as a product on promotion or a customer eligible for a campaign. Anti-join it against the activity fact to answer "what did not happen".

Keep the same dimension keys and the same grain as the activity fact; comparing a monthly coverage fact against daily sales makes every product look unsold on most days.

In [ ]:
con.execute("""
create table fact_promotion_coverage (product_sk bigint, date_sk int, primary key (product_sk, date_sk));
insert into fact_promotion_coverage values (1, 20261015), (2, 20261015), (3, 20261015);
""")

unsold = con.execute("""
    select c.product_sk
    from fact_promotion_coverage c
    left join fact_order_line f on f.product_sk = c.product_sk and f.order_date_sk = c.date_sk
    where f.product_sk is null
""").fetchall()
print("promoted but unsold on 2026-10-15:", unsold)
assert unsold == [(3,)]

## What is a date dimension and a role-playing dimension?

A **date dimension** has one row per calendar day with precomputed attributes the business argues about: fiscal period, week start, `is_weekend`, `is_holiday`, the same day last year. Use an integer `date_sk` in `YYYYMMDD` form and reserve members for unknown and not-applicable dates. Without it, timezone and fiscal rules end up re-derived in every dashboard.

A **role-playing dimension** is one physical dimension joined several times with different meanings: `order_date_sk`, `ship_date_sk`, `delivery_date_sk` all point at `dim_date`; origin and destination both point at `dim_airport`. Join it with aliases (or thin views per role); never build physical copies that drift when a holiday is added to one.

```sql
select dd.year_month as order_month, ds.year_month as ship_month, count(*) as orders
from fact_order f
join dim_date dd on dd.date_sk = f.order_date_sk
join dim_date ds on ds.date_sk = f.ship_date_sk
group by 1, 2;
```

A date dimension is generated, not loaded from a source: one row per day for a fixed range, so queries never apply functions to the fact's date column. The cell below generates eleven years with weekend and April-start fiscal-year attributes; the small `dim_date` built at the start is the same idea over two months.

In [ ]:
con.execute("""
CREATE TABLE dim_date_full AS
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER)          AS date_sk,
       CAST(d AS DATE)                                 AS date,
       year(d)                                         AS year,
       quarter(d)                                      AS quarter,
       strftime(d, '%b')                               AS month_name,
       isodow(d) IN (6, 7)                             AS is_weekend,
       CASE WHEN month(d) >= 4 THEN year(d) ELSE year(d) - 1 END AS fiscal_year  -- April fiscal start
FROM generate_series(DATE '2020-01-01', DATE '2030-12-31', INTERVAL 1 DAY) AS t(d)
""")
rows, weekends = con.sql("SELECT COUNT(*), COUNT(*) FILTER (WHERE is_weekend) FROM dim_date_full").fetchall()[0]
assert rows == 4018  # 11 years including 3 leap days
assert con.sql("SELECT fiscal_year FROM dim_date_full WHERE date = DATE '2026-02-01'").fetchall()[0][0] == 2025
print(f"{rows} days, {weekends} weekend days")

## What is a metric definition layer and why does it matter?

Most metric disputes are modeling problems, not SQL problems. A metric contract (dbt semantic layer, Cube, LookML, or a governed table) pins down the expression, grain, filters, time dimension, allowed dimensions, owner, and additivity, so three dashboards cannot ship three revenue numbers.

```yaml
metric: net_revenue
owner: finance-analytics
base_table: gold.fact_order_line
grain: order_line
expression: sum(gross_amount - discount_amount - refund_amount)
filters: ["order_status <> 'cancelled'"]
time_dimension: order_date
allowed_dimensions: [dim_customer.segment, dim_customer.country_code, dim_product.category]
currency: USD
```

Additivity rules for contracts: ratios store numerator and denominator (`average_order_value = sum(net_revenue) / count(distinct order_id)`), distinct counts name the exact key, semi-additive measures name their time rollup. Averaging averages weights each row equally regardless of how many events it summarizes, so it is wrong whenever groups differ in size.

## Kimball, Inmon, or Data Vault: which approach should you use?

| Approach | First artifact | Strength | Cost |
|---|---|---|---|
| Kimball (bottom-up) | dimensional mart per business process, integrated by conformed dimensions and a bus matrix | fast time to value, simple queries | integration discipline needed later |
| Inmon (top-down) | normalized (3NF) enterprise warehouse: subject-oriented, integrated, non-volatile, time-variant; marts derived from it | one version of the truth | long initial build |
| Data Vault | hubs (business keys), links (relationships), satellites (descriptive history), insert-only | auditability, onboarding many changing sources | more tables; needs marts on top for BI |

They compose rather than compete. The pragmatic hybrid is the usual interview answer: land raw data (bronze), normalize and conform in silver (Inmon's discipline, or Data Vault for many-source parallel ingestion and audit), publish dimensional marts in gold (Kimball's delivery). Frame the choice by constraints, not ideology: auditors and many incrementally arriving sources favor a normalized or vault core; time to value and self-service favor dimensional marts; a small, stable set of sources does not need a vault. Building marts straight from sources before entities are conformed leads each mart to invent its own customer.

## What is a Data Vault model?

Data Vault 2.0 is an insert-only integration pattern built from three constructs:

+ **Hubs**: one row per business key ever seen, keyed by a hash of the key.
+ **Links**: relationships between hubs (order to customer).
+ **Satellites**: descriptive attributes with history (`load_ts`, `record_source`, `hash_diff` over tracked attributes) attached to a hub or link.

Every load is keyed by hash, so reruns insert nothing new, sources load in parallel, and `hash_diff` detects change without comparing every column. The cost: one entity is scattered across many tables, so BI directly on raw vault is painful. Teams publish point-in-time (PIT) and bridge tables, then dimensional marts on top. Data Vault is an integration layer, not a reporting model. Gotcha: the same attribute in two satellites (or in the hub and a satellite) makes change detection ambiguous.

Satellites insert a row only when the descriptive hash changes:

```sql
INSERT INTO sat_customer
SELECT s.customer_hk, s.load_ts, s.hash_diff, s.name, s.city, s.segment
FROM staging.customer s
LEFT JOIN sat_customer c ON c.customer_hk = s.customer_hk AND c.hash_diff = s.hash_diff
WHERE c.customer_hk IS NULL;
```

The cell below loads a hub twice from a staging table that contains a duplicate, and shows the rerun inserts nothing.

In [ ]:
con.execute("""
create table hub_customer (customer_hk varchar primary key, customer_id varchar, load_ts timestamp, record_source varchar);
create table stg_crm_customer (customer_id varchar, load_ts timestamp);
insert into stg_crm_customer values ('C1', '2026-10-01'), ('C2', '2026-10-01'), ('C1', '2026-10-01');
""")


def load_hub():
    con.execute("""
        insert into hub_customer
        select md5(customer_id), customer_id, min(load_ts), 'crm'
        from stg_crm_customer s
        where not exists (select 1 from hub_customer h where h.customer_hk = md5(s.customer_id))
        group by customer_id
    """)
    return con.execute("select count(*) from hub_customer").fetchone()[0]


first_load, rerun = load_hub(), load_hub()
print("hub rows after load:", first_load, "| after rerun:", rerun)
assert first_load == rerun == 2   # duplicates collapsed, rerun idempotent

## What are the layers of a data warehouse?

Each layer changes the data in exactly one way, so a bug can be fixed by rebuilding one layer instead of re-reading
the source.

```text
source -> landing/raw -> staging -> ODS -> warehouse (core) -> marts -> BI
          append-only    typed,     current   conformed dims    aggregated,
          replayable     deduped    entities  atomic facts      per team
```

+ **Landing / raw (bronze)**: an immutable copy of what the source sent, so a bad parse can be replayed.
+ **Staging (silver)**: typed and deduplicated; the only place raw payloads are parsed.
+ **ODS**: current-state operational entities, integrated across systems, little history.
+ **Warehouse / integration**: business rules, conformed dimensions, atomic facts, history.
+ **Presentation / marts (gold)**: star schemas, aggregates, and the semantic layer for one team or tool.

Worked example: an order arrives twice because of a retry. Landing keeps both rows, staging deduplicates on
`order_id` and the update timestamp, the ODS shows the latest status, the warehouse keeps the order fact, and the mart
shows today's open orders.

Common mistake: business joins in staging. A rule change then forces a re-read of raw data instead of a rebuild of
one model layer.

The classic **three-tier** description maps onto the same layers: bottom tier = sources, ingestion, and storage;
middle tier = integration, warehouse, and semantic model; top tier = BI, ad hoc SQL, and APIs.

## How do OLTP and OLAP systems differ?

| | OLTP | OLAP |
|---|---|---|
| Workload | many small reads/writes (place an order) | large scans and aggregations (revenue by month) |
| Model | normalized (3NF), ER modeling | dimensional (star/snowflake) |
| Storage | row-oriented, B-tree indexes | columnar, partition pruning, zone maps |
| Priority | low-latency writes, consistency | scan throughput, concurrency of reads |

Data moves from OLTP to OLAP with batch extracts, CDC, or event streams.

## How fresh can a warehouse be?

| Mode | Freshness | Typical stack | Cost model |
|---|---|---|---|
| Batch | hours to a day | nightly ELT | per run |
| Micro-batch | 1-15 minutes | Spark Structured Streaming, Auto Loader | per run, many runs |
| Near real time | seconds | Kafka + Flink + a fast store | per running hour |
| Real time | sub-second | streaming engine + serving layer | per running hour |

A true real-time warehouse fights the reason a warehouse exists (isolating analytics from operations) and makes
surrogate-key resolution and aggregate maintenance hard. Near real time is the usual compromise: frequent selective
refreshes of the tables that change, with full rebuilds further apart. Every streaming path needs an event-time
watermark, a dedup key, and a batch reconciliation, or it drifts away from the warehouse.

## What are the medallion architecture and the lakehouse?

**Medallion** is the lakehouse name for the same layering: bronze (raw, append-only, replayable) -> silver (typed,
deduplicated, conformed keys) -> gold (star schemas, marts, aggregates). Rules: bronze is append-only so anything
downstream can be rebuilt; only silver parses and deduplicates; each layer has an owner, a freshness SLO, and a schema
contract; never land straight into gold.

A **lakehouse** is object storage + an open table format (Delta, Iceberg, Hudi) + a catalog: warehouse semantics on
lake storage.

| Aspect | Data warehouse | Lakehouse |
|---|---|---|
| Storage | proprietary, managed | open files (Parquet) plus table format |
| Compute | coupled to one engine | pluggable (Spark, Trino, Flink, SQL) |
| Transactions | native | table-format ACID, snapshots, time travel |
| Data types | structured, some semi-structured | structured, semi- and unstructured |
| Governance | mature RBAC and lineage | needs a catalog |

Choose a lakehouse for open formats and many engines; choose a managed warehouse for high-concurrency BI with less
operational overhead. The format adds reliability to storage, not modeling: grain, keys, and tests still apply.

## What are data marts, ODS, and virtual warehouses?

+ **Data mart**: a subject-area subset of the warehouse (finance, loans) for one team or tool; a different scope, not
  a different technology. **Dependent** marts are fed from the warehouse and inherit its conformed dimensions;
  **independent** marts read sources directly and drift into a second definition of revenue.
+ **ODS (operational data store)**: integrated, current-state, near-real-time operational data with short history.
  A mart can be built from an ODS when it only needs recent operational data, but historical analysis needs the
  warehouse.
+ **Virtual data warehouse**: users query operational databases directly (federation) instead of a physical copy.
  Cheap to start; slow and risky for the source systems at scale.

## ETL versus ELT: where should transformation run?

ETL transforms on a separate server before loading; ELT loads raw data first and transforms inside the warehouse with
SQL models. ELT wins on scalable cloud warehouses because raw data stays replayable: a wrong join means rebuilding one
model, not re-extracting from the source. ETL still fits constrained targets or data that must be masked before it
lands. ELT moves the work into SQL, so layering, tests, orchestration, and idempotency matter just as much.

### Staging: type and deduplicate

Staging keeps the latest version of each business key with `ROW_NUMBER()` (or DuckDB/Snowflake/BigQuery `QUALIFY`).

In [ ]:
con.execute("""
CREATE TABLE landing_orders (order_id INTEGER, status TEXT, amount TEXT, updated_at TIMESTAMP);
INSERT INTO landing_orders VALUES
    (1, ' paid ',   '10.50', TIMESTAMP '2026-01-05 10:00'),
    (1, 'shipped',  '10.50', TIMESTAMP '2026-01-06 09:00'),   -- later version of order 1
    (1, 'shipped',  '10.50', TIMESTAMP '2026-01-06 09:00'),   -- exact duplicate from a retry
    (2, 'paid',     '99.00', TIMESTAMP '2026-01-06 11:00');

CREATE TABLE stg_orders AS
SELECT order_id, UPPER(TRIM(status)) AS status, CAST(amount AS DECIMAL(12, 2)) AS amount, updated_at
FROM landing_orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1;
""")
staged = con.execute("SELECT * FROM stg_orders ORDER BY order_id").fetchall()
for row in staged:
    print(row)
assert [(row[0], row[1]) for row in staged] == [(1, "SHIPPED"), (2, "PAID")]

## Full load versus incremental load

A **full load** truncates and rebuilds the target (simple, right for small dimensions; build a new table and swap it
atomically). An **incremental load** applies only changed rows since a stored **watermark** and must be idempotent: an
upsert on the natural key, or delete-the-partition-then-insert. A plain append is not idempotent: a retry duplicates
the partition.

(Database-specific "conventional" versus "direct-path" loads are the same trade-off at the engine level: validate
constraints while inserting, or disable them for speed and validate afterwards.)

In [ ]:
con.execute("""
CREATE TABLE fact_order_status (order_id INTEGER PRIMARY KEY, status TEXT, amount DECIMAL(12, 2),
                               updated_at TIMESTAMP);
CREATE TABLE etl_watermark (table_name TEXT PRIMARY KEY, high_water TIMESTAMP);
INSERT INTO etl_watermark VALUES ('fact_order_status', TIMESTAMP '1900-01-01');
""")

UPSERT = """
INSERT INTO fact_order_status
SELECT order_id, status, amount, updated_at FROM stg_orders
WHERE updated_at > (SELECT high_water FROM etl_watermark WHERE table_name = 'fact_order_status')
ON CONFLICT (order_id) DO UPDATE
    SET status = excluded.status, amount = excluded.amount, updated_at = excluded.updated_at
    WHERE excluded.updated_at > fact_order_status.updated_at;

UPDATE etl_watermark
SET high_water = (SELECT MAX(updated_at) FROM fact_order_status)
WHERE table_name = 'fact_order_status';
"""
con.execute(UPSERT)
con.execute(UPSERT)  # a retry reads nothing new and changes nothing

con.execute("INSERT INTO stg_orders VALUES (2, 'REFUNDED', 99.00, TIMESTAMP '2026-01-07 08:00')")
con.execute(UPSERT)  # the next run picks up only the new change
final = con.execute("SELECT order_id, status, updated_at FROM fact_order_status ORDER BY order_id").fetchall()
for row in final:
    print(row)
assert [(order_id, status) for order_id, status, _ in final] == [(1, "SHIPPED"), (2, "REFUNDED")]

## How do point-in-time (as-of) joins resolve dimension keys for facts?

A fact row resolves to the version whose range contains the event time: `event_ts >= valid_from and event_ts < valid_to`. The half-open range guarantees adjacent versions never both match. Using `between valid_from and valid_to` makes the boundary instant match two versions and duplicates the fact.

Two ways to apply it:

+ **Resolve at load time**: look up the surrogate key when the fact is loaded, so reports join on a single equi-key. Best when facts arrive roughly in order.
+ **Resolve at query time**: range join on natural key plus validity window. Put the equality on the natural key first so it is not a cross product, and prune history by partition.

Don't use `is_current = true` as a shortcut for "as of" in a report that is rerun historically: next month's rerun returns a different answer for the same period, which fails audit.

The cell below resolves each order line to the customer version valid at order time, proves the half-open join keeps the fact row count, and then uses the Type 6 table to report revenue by the segment at order time next to the segment the customer is in now.

In [ ]:
as_of = con.execute("""
    select f.order_id, f.line_number, f.order_ts, c.customer_sk, c.segment as segment_at_order
    from fact_order_line f
    join dim_customer c
      on c.customer_id = f.customer_id
     and f.order_ts >= c.valid_from
     and f.order_ts <  c.valid_to
    order by f.order_id, f.line_number
""").fetchall()
for row in as_of:
    print(row)

# exactly one version per fact row: the join preserves the fact row count
assert len(as_of) == 5
segments = {(r[0], r[1]): r[4] for r in as_of}
assert segments[("O1", 1)] == "standard"      # September order -> old version
assert segments[("O3", 1)] == "enterprise"    # October order -> new version

# inclusive BETWEEN double-matches an event exactly on the version boundary
boundary = con.execute("""
    select count(*) from dim_customer
    where customer_id = 'C1' and timestamp '2026-10-01' between valid_from and valid_to
""").fetchone()[0]
assert boundary == 2

# Type 6 payoff: the same as-of join reports by the then-true and the current segment
then_vs_now = con.execute("""
    select c.segment as segment_at_order, c.segment_current, sum(f.net_amount) as revenue
    from fact_order_line f
    join dim_customer_t6 c
      on c.customer_id = f.customer_id
     and f.order_ts >= c.valid_from
     and f.order_ts <  c.valid_to
    group by all
    order by all
""").fetchall()
for row in then_vs_now:
    print(row)
assert then_vs_now == [
    ("enterprise", "enterprise", 420),
    ("standard", "enterprise", 180),   # C1's September order, before the upgrade
    ("standard", "standard", 200),
]

## How do you handle late-arriving dimensions and facts?

Late data is normal: a fact for 2026-09-15 lands on 2026-10-04, or a customer appears in sales before the customer feed has them.

+ **Late fact, dimension known**: write the fact to the partition for its *event* date and resolve the dimension as of the event time. Never use load time as event time; it shifts metrics when pipelines are delayed and breaks partition pruning.
+ **Late fact, dimension unknown** (an early-arriving fact): never drop the fact. The simplest fix points it at an **unknown member** (`customer_sk = -1`), keeps the natural key on the fact, and repairs the key when the real dimension row arrives. The richer fix inserts an **inferred member** (natural key, placeholder attributes, `is_inferred = true`) so the fact gets its own key now and the next SCD load fills in the attributes. Either way, track how many unresolved rows remain.
+ **Late dimension change effective before already-loaded facts**: an out-of-order version. You must split the covering version's range and possibly re-point fact rows, which is a restatement. Store the source effective date and handle it through a controlled backfill rather than silently rewriting reported history.

Control the blast radius with a restatement window: accept late data for an agreed window (for example the last 7 days) and recompute only the affected partitions and aggregates; older corrections go through a documented backfill. Don't create a separate "late data" table: it doubles every downstream query and hides the gap.

```sql
-- inferred member: facts can join immediately, attributes filled in later
insert into dim_customer (customer_sk, customer_id, segment, valid_from, valid_to, is_current, is_inferred)
select nextval('customer_sk_seq'), s.customer_id, 'UNKNOWN', timestamp '1900-01-01', timestamp '9999-12-31', true, true
from (select distinct customer_id from stg_order) s
where not exists (select 1 from dim_customer d where d.customer_id = s.customer_id);
```

The cell below runs the unknown-member pattern end to end: the fact lands with key `-1`, the customer feed catches up, and the key is repaired as of the order time.

In [ ]:
con.execute("""
insert into dim_customer values (-1, 'UNKNOWN', 'unknown', null, '1900-01-01', '9999-12-31', true);

create table stg_late_order (order_id varchar, customer_id varchar, order_ts timestamp, net_amount decimal(14, 2));
insert into stg_late_order values ('O9', 'C4', '2026-10-20 12:00', 75.00);   -- C4 is not in dim_customer yet

create table fact_order_customer as
select s.order_id, s.customer_id, s.order_ts, coalesce(d.customer_sk, -1) as customer_sk, s.net_amount
from stg_late_order s
left join dim_customer d
  on d.customer_id = s.customer_id and s.order_ts >= d.valid_from and s.order_ts < d.valid_to;
""")
assert con.execute("select customer_sk from fact_order_customer").fetchone()[0] == -1   # kept, not dropped

# the customer feed catches up (C4 effective 2026-10-15): load it, then repair the fact key as of the order
load_scd2([("C4", "standard", "c4@x.com", "2026-10-15 08:00")], "2026-10-15")
con.execute("""
    update fact_order_customer f set customer_sk = d.customer_sk
    from dim_customer d
    where f.customer_sk = -1 and d.customer_id = f.customer_id
      and f.order_ts >= d.valid_from and f.order_ts < d.valid_to
""")
repaired = con.execute("""
    select f.order_id, d.customer_id, d.segment
    from fact_order_customer f join dim_customer d using (customer_sk)
""").fetchall()
print("repaired:", repaired)
assert repaired == [("O9", "C4", "standard")]

## What are metadata, the catalog, and lineage?

Metadata is data about data. **Technical** metadata: schemas, types, partitions, freshness. **Business** metadata:
definitions, owners, grain. **Operational** metadata: load times, row counts, lineage. A data dictionary is the
subset that describes columns and valid values. The catalog records what exists and who owns it; column-level lineage
records where each column came from and what depends on it, which turns "what breaks if I change this column?" into
a query instead of an outage. A catalog must be generated from the running system, or it is stale within a sprint.

## How do you design for performance and scale?

Performance is mostly decided by four modeling choices: grain, denormalization, physical layout, and SCD history volume.

### Pre-aggregation, views, and materialized views

A **view** stores only the query and recomputes it on every read. A **materialized view** stores the result and must be refreshed (incrementally where the engine supports it). An **aggregate table** is a summary at a coarser grain built by the pipeline; it must be refreshed and tested like any other output.

Pre-aggregate when the same heavy group-by runs constantly and its result is small (500M order lines by category and month is ~48K rows). Keep the base fact at event grain; summaries are derived, named by grain (`fact_sales_monthly_category`), and store additive components only. A summary holding `avg_order_value` cannot be rolled up to a year correctly; store `sum_revenue` and `order_count` and divide at query time.

### Physical layout

+ **Partition** by event date or tenant so filters prune files, and keep partitions large (hundreds of MB to a few GB). Never partition by a high-cardinality key like `order_id` or `customer_id` (millions of tiny files).
+ **Cluster / sort** within partitions by common filter and join keys so min/max statistics (zone maps) skip blocks.
+ Keep files in a healthy size range and compact small files.
+ Align the model's join key with the physical sort key: joining on `customer_id` strings when the table is clustered on `customer_sk` cannot use the sort order.
+ **Indexes**: columnar warehouses rely on pruning, zone maps, and bloom filters rather than B-trees. Row-store warehouses use bitmap indexes on low-cardinality fact foreign keys and flags (bitmaps suit low cardinality, such as 0/1 flags) and B-trees on high-cardinality dimension surrogate and natural keys.

```sql
-- Snowflake-style clustering on the columns the model filters and joins on
alter table fact_order_line cluster by (order_date_sk, customer_sk);
-- Iceberg: change the partition spec without rewriting history
alter table fact_order_line set partition spec (month(order_ts));
```

### SCD2 at scale

History grows with change rate, not entity count: 12M customers changing 2% per day add 240K rows/day, about 88M/year. Keep it manageable by partitioning history by `valid_from`, exposing a small current table or view, merging on the natural key before the range predicate, and archiving closed versions beyond retention. Full rebuilds every run eventually stop fitting in the batch window.

### MOLAP, ROLAP, HOLAP

The older OLAP taxonomy split engines by where aggregates live: MOLAP precomputes a multidimensional cube (fastest queries, slow builds, limited detail), ROLAP runs SQL over a relational star at query time, and HOLAP keeps detail relational and aggregates in a cube. Modern columnar warehouses are effectively ROLAP with materialized views and result caches.

## How do hierarchies, ROLLUP, and GROUPING SETS work?

A hierarchy is an ordered set of levels inside a dimension (department -> category -> product, year -> quarter -> month). Storing every level on the dimension row (star style) makes drill-down a `GROUP BY`. `ROLLUP` produces subtotals along the hierarchy, `GROUPING SETS` picks arbitrary combinations in one pass, and `GROUPING()` tells a subtotal row from a real NULL. Precomputing these combinations is what OLAP aggregations are; choosing which ones to materialize is the **view selection problem**.

Distinct counts do not re-aggregate from a coarser table: keep them at the detail grain or use a sketch such as HyperLogLog.

In [ ]:
rollup = con.execute("""
    select p.department, p.category, sum(f.net_amount) as revenue,
           grouping(p.department, p.category) as level
    from fact_order_line f
    join dim_product p using (product_sk)
    group by rollup (p.department, p.category)
    order by level, p.department, p.category
""").fetchall()
for row in rollup:
    print(row)
assert [revenue for _, _, revenue, level in rollup if level == 3] == [800]   # grand total
assert {(dept, revenue) for dept, _, revenue, level in rollup if level == 1} == {("Home", 60), ("Outdoor", 740)}

# GROUPING SETS: only the combinations a dashboard needs, here department totals and month totals
sets = con.execute("""
    select p.department, d.year_month, sum(f.net_amount) as revenue
    from fact_order_line f
    join dim_product p using (product_sk)
    join dim_date d on d.date_sk = f.order_date_sk
    group by grouping sets ((p.department), (d.year_month))
    order by all
""").fetchall()
for row in sets:
    print(row)
assert len(sets) == 4   # 2 departments + 2 months

## How do you test a warehouse model after changes?

Every test is an invariant: a query that must return zero rows, run as a pipeline step that fails the load rather than logging a warning. Test at four levels:

| Level | Example test | Fails when |
|---|---|---|
| Schema | column exists and type matches | the source changes shape |
| Grain | declared key is unique | a loader duplicates a row |
| Integrity | every foreign key resolves; one current row per natural key; no overlapping validity ranges | a late dimension row is missing |
| Business | mart totals reconcile with the source within a tolerance; no unexplained date or entity gaps | a join silently drops or duplicates rows |

Row counts alone miss wrong joins that change values but not cardinality, so reconcile measures, not just counts. Test known edges too: a customer with no segment, a fact with a null key, a duplicate feed file. A test that only warns is a test that will be ignored. The cell below runs these checks against everything built above.

In [ ]:
TESTS = {
    "grain_unique": """select order_id, line_number from fact_order_line
                       group by all having count(*) > 1""",
    "one_current_version": """select customer_id from dim_customer
                              group by customer_id having count(*) filter (where is_current) <> 1""",
    "valid_ranges": """select customer_sk from dim_customer where valid_to <= valid_from""",
    "no_overlapping_versions": """select a.customer_id from dim_customer a join dim_customer b
                                  on a.customer_id = b.customer_id and a.customer_sk < b.customer_sk
                                  and a.valid_from < b.valid_to and b.valid_from < a.valid_to""",
    "no_orphan_products": """select f.product_sk from fact_order_line f
                             left join dim_product p using (product_sk) where p.product_sk is null""",
    "no_unresolved_customers": """select order_id from fact_order_customer where customer_sk = -1""",
    "order_totals_reconcile": """select 1 where (select sum(revenue) from fact_order)
                                 <> (select sum(net_amount) from fact_order_line)""",
}
results = {name: len(con.execute(sql).fetchall()) for name, sql in TESTS.items()}
for name, failures in results.items():
    print(f"{'PASS' if failures == 0 else 'FAIL'}  {name}")
assert all(failures == 0 for failures in results.values())

## Designing a warehouse that scales

Four decisions, made in order:

1. **Architecture.** A layered warehouse when questions are known and governance is a hard requirement; a lakehouse
   when files are already the source of truth or one copy must serve many engines; a data mesh (domain-owned data
   products on a central platform) only when many teams own many sources and contracts plus a platform team exist.
   Central modeling gives one metric definition but is a bottleneck; decentralization gives speed but multiplies
   definitions. Pick one system of record per dataset.
2. **Ingestion.** Batch cost scales with runs, streaming cost with wall-clock hours (idle capacity at night).
   Micro-batch is the usual analytics default; hybrid adds a fast path plus a nightly reconciliation.
3. **Transformation.** ELT with orchestration (dependency order, retries, backfills), tests that fail the run, and
   idempotent loads. Schedule by data interval, not completion time, so backfills do not collide with live runs; reload
   a bounded late-data window instead of all history.
4. **Operations.** Publish a freshness SLA per table and measure event time versus availability, not job duration
   (a late start with a fast job is still stale). Watch the largest cost consumers and idle compute, capture
   column-level lineage for impact analysis, and prove RPO/RTO with restore drills.

```yaml
pipeline: daily_sales
schedule: "0 2 * * *"
tasks:
  - {id: stage_orders, depends_on: []}
  - {id: dim_customer_scd2, depends_on: [stage_orders]}
  - {id: fct_sales, depends_on: [dim_customer_scd2]}
  - {id: test_reconcile, depends_on: [fct_sales]}
```

## Real world use cases for data modeling

| Domain | Pattern | Why |
|---|---|---|
| Retail, ~40M order lines/month | order-line star + weighted promotion bridge | margin by category/store/promotion without double counting; conformed date and product |
| Fintech, 7-year audit | SCD2 customer dim with as-of joins on transaction time | regulators need the risk tier that was true at the time; Type 1 would rewrite history |
| B2B SaaS, 2,000 tenants | `tenant_sk` on every fact and dimension + row-level security | partition pruning and isolation; cross-tenant joins can never match |
| Marketing | factless eligibility fact anti-joined to conversions | "eligible but did not convert" is about a relationship, not a measure |
| Logistics | accumulating snapshot per shipment | stage durations become `avg(days_in_transit)` instead of event-stream scans |

```sql
-- fintech: attribute values as of each transaction
select t.transaction_id, c.risk_tier as risk_tier_at_transaction, t.amount
from fact_transaction t
join dim_customer c
  on c.customer_id = t.customer_id
 and t.transaction_ts >= c.valid_from
 and t.transaction_ts <  c.valid_to;
```

## Real-world use case: clickstream sessionization

A new session starts after 30 minutes of inactivity. `LAG` finds the gap, a running `SUM` numbers the sessions, and the
result becomes a session fact that joins to the revenue fact on user and time.

In [ ]:
con.execute("""
CREATE TABLE click_events (user_id TEXT, event_ts TIMESTAMP);
INSERT INTO click_events VALUES
    ('u1', TIMESTAMP '2026-01-05 09:00'), ('u1', TIMESTAMP '2026-01-05 09:10'),
    ('u1', TIMESTAMP '2026-01-05 09:55'), ('u1', TIMESTAMP '2026-01-05 10:05'),
    ('u2', TIMESTAMP '2026-01-05 09:00');
""")
sessions = con.execute("""
WITH flagged AS (
    SELECT *,
           CASE WHEN event_ts - LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) <= INTERVAL 30 MINUTE
                THEN 0 ELSE 1 END AS is_new_session
    FROM click_events
), numbered AS (
    SELECT *, SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts) AS session_number
    FROM flagged
)
SELECT user_id, session_number, MIN(event_ts) AS session_start, MAX(event_ts) AS session_end, COUNT(*) AS events
FROM numbered
GROUP BY user_id, session_number
ORDER BY user_id, session_number
""").fetchall()
for row in sessions:
    print(row)
assert [(user, number, events) for user, number, _, _, events in sessions] == [("u1", 1, 2), ("u1", 2, 2), ("u2", 1, 1)]

## Things to remember

| Topic | Fact | Why it matters |
|---|---|---|
| Grain | "one row per ___" is decided before any column | every double-count bug is a grain bug |
| SCD types | 0 fixed, 1 overwrite, 2 new row with dates, 3 prior-value column, 6 = 2 + current column | Type 2 is the default answer and the expensive one |
| Star vs snowflake | star denormalizes dimensions | on MPP engines fewer joins win |
| Surrogate keys | warehouse-generated, decoupled from source keys | Type 2 is impossible without them |
| Conformed dimensions | the same dimension across marts | keeps metrics comparable; enables drill-across |
| Kimball vs Inmon | marts first via a bus vs normalized EDW first | Kimball ships faster; Inmon integrates first |
| Medallion | bronze -> silver -> gold | staging + warehouse + mart, restated for lakehouses |
| Fact types | transaction, periodic snapshot, accumulating snapshot, factless | the type decides what late data means |

Mistakes that keep coming back:

+ Declaring the grain after the columns exist.
+ Type 2 on every attribute by default (row explosion), or Type 2 on corrections (fake history).
+ Source primary keys as warehouse keys: a source re-key breaks history.
+ Joining two fact tables row to row instead of aggregating each to a shared grain first.
+ A "conformed" dimension that differs slightly per mart, so metrics diverge silently.
+ No unknown member for late dimensions: facts land with dangling keys and vanish from reports.
+ Appending in a retried load instead of upserting or replacing the partition.

## Data modeling and warehousing exercises

Small standard-library calculations behind common modeling and warehouse decisions: grain, fan-out, history growth, storage, cost, freshness, and idempotency. Each states the task and the expected result, then asserts it. The numbers check the formula, not a real platform: compression, skew, caching, and concurrency add variance.

### Try it 1. Pick a grain from a requirement

**Task.** Find the single finest grain that answers all requirements. A requirement is answerable from its required level or finer (lower number). **Expected:** `order_line`.

In [ ]:
GRANULARITY = {"order_month": 3, "order": 2, "order_line": 1}
REQUIREMENTS = {
    "revenue by product by month": 2,
    "average order value": 2,
    "units per customer per day": 1,
}

finest_required = min(REQUIREMENTS.values())
valid = sorted(g for g, level in GRANULARITY.items() if level <= finest_required)
chosen = min(valid, key=lambda g: GRANULARITY[g])

print("finest level required:", finest_required, "| chosen grain:", chosen)
assert chosen == "order_line"

### Try it 2. Detect fan-out from join cardinalities

**Task.** Joined rows are `fact_rows_per_key x dim_rows_per_key` summed over keys; the measure inflates by the same ratio. **Expected:** 1,800 rows, 1.8x, 90,000 from a true 50,000.

In [ ]:
fact = {"total_rows": 1000, "total_amount": 50000}
fact_rows_per_key = {1: 400, 2: 400, 3: 200}
dim_rows_per_key = {1: 2, 2: 1, 3: 3}

joined_rows = sum(fact_rows_per_key[k] * dim_rows_per_key[k] for k in fact_rows_per_key)
ratio = joined_rows / fact["total_rows"]
inflated = fact["total_amount"] * ratio

print(f"joined rows = {joined_rows} ({ratio:.2f}x), inflated measure = {inflated:.0f}")
assert joined_rows == 1800 and ratio == 1.8 and inflated == 90000

### Try it 3. Star or snowflake from query patterns

**Task.** Compute weighted average join hops per query for each shape. **Expected:** star 1.0 vs snowflake 2.4, so star.

In [ ]:
QUERY_PATTERNS = {"single_dimension": 700, "two_hops": 200, "three_hops": 100}
STAR_HOPS = {"single_dimension": 1, "two_hops": 1, "three_hops": 1}
SNOWFLAKE_HOPS = {"single_dimension": 2, "two_hops": 3, "three_hops": 4}
total_queries = sum(QUERY_PATTERNS.values())


def weighted_hops(hops):
    return sum(QUERY_PATTERNS[p] * hops[p] for p in QUERY_PATTERNS) / total_queries


star_hops, snowflake_hops = weighted_hops(STAR_HOPS), weighted_hops(SNOWFLAKE_HOPS)
print(f"star = {star_hops:.2f}, snowflake = {snowflake_hops:.2f}")
assert round(star_hops, 2) == 1.0 and round(snowflake_hops, 2) == 2.4

### Try it 4. SCD2 history growth

**Task.** 12M customers, 2% change per day, 7-year retention. **Expected:** 240,000 rows/day, 87.6M/year, 613.2M retained.

In [ ]:
entities, change_rate, retention_years = 12_000_000, 0.02, 7

rows_per_day = entities * change_rate
rows_per_year = rows_per_day * 365
rows_retained = rows_per_year * retention_years

print(f"{rows_per_day:,.0f}/day, {rows_per_year:,.0f}/year, {rows_retained:,.0f} retained")
assert rows_per_day == 240_000 and rows_per_year == 87_600_000 and rows_retained == 613_200_000

### Try it 5. Size a bridge table

**Task.** 8M orders/month; 15% carry two promotions, 5% carry three. **Expected:** 0.45 relationships per order, 3.6M bridge rows/month, 43.2M/year.

In [ ]:
orders_per_month = 8_000_000
PROMOTIONS_PER_ORDER = {2: 0.15, 3: 0.05}

relationships_per_order = sum(count * share for count, share in PROMOTIONS_PER_ORDER.items())
bridge_rows_month = orders_per_month * relationships_per_order

print(f"{relationships_per_order:.2f} per order, {bridge_rows_month:,.0f}/month")
assert round(relationships_per_order, 2) == 0.45
assert round(bridge_rows_month) == 3_600_000 and round(bridge_rows_month * 12) == 43_200_000

### Try it 6. Detect a mixed grain

**Task.** Order-level shipping is stored on every order line. Compare the true total with the line-grain sum. **Expected:** 35 vs 115 (3.29x over-count).

In [ ]:
order_lines = {"O1": 3, "O2": 1, "O3": 4}
shipping_per_order = {"O1": 10, "O2": 5, "O3": 20}

true_total = sum(shipping_per_order.values())
naive_total = sum(shipping_per_order[o] * n for o, n in order_lines.items())

print(f"true = {true_total}, summed at line grain = {naive_total} ({naive_total / true_total:.2f}x)")
assert true_total == 35 and naive_total == 115

### Try it 7. Size storage from daily volume and retention

**Task.** A fact table receives 50,000,000 rows per day at 400 bytes per row before compression. The warehouse compresses at 4 to 1. Add 15 percent for indexes and metadata, keep two copies for redundancy, and retain 730 days. Compute the stored bytes per day and the total footprint in terabytes.

**Hint.** Convert to gigabytes first: `daily_rows * bytes_per_row / 1e9`. Then divide by compression, multiply by overhead and copies, and multiply by the retention days.

**Expected result.** `20.0 GB` raw per day, `11.5 GB` stored per day, and about `8395.0 GB` or `8.395` TB for the full retention.

In [ ]:
DAILY_ROWS = 50_000_000
BYTES_PER_ROW = 400
COMPRESSION = 4
OVERHEAD = 1.15
COPIES = 2
RETENTION_DAYS = 730

raw_gb_per_day = DAILY_ROWS * BYTES_PER_ROW / 1e9
stored_gb_per_day = raw_gb_per_day / COMPRESSION * OVERHEAD * COPIES
total_gb = stored_gb_per_day * RETENTION_DAYS

print(f"raw          = {raw_gb_per_day:.1f} GB/day")
print(f"stored       = {stored_gb_per_day:.1f} GB/day")
print(f"{RETENTION_DAYS} days     = {total_gb:.1f} GB = {total_gb / 1000:.3f} TB")

assert raw_gb_per_day == 20.0
assert stored_gb_per_day == 11.5
assert round(total_gb, 1) == 8395.0
assert round(total_gb / 1000, 3) == 8.395

### Try it 8. Compute transformation cost from data volume and engine rate

**Task.** A nightly transformation reads 2000 GB. Each of 8 worker nodes processes 5 GB per minute. Shuffle and write overhead adds 25 percent to the billed runtime, and a node-hour costs 6.00. Compute the full-refresh cost, then the cost when an incremental run reads only 5 percent of the data.

**Hint.** `runtime_minutes = total_gb / (rate_per_node * nodes)`. Billed node-hours are `nodes * runtime_minutes * overhead / 60`.

**Expected result.** Full refresh runs `50` minutes, bills `8.333` node-hours, and costs `50.0`. The incremental run costs `2.5`, a 20 times reduction.

In [ ]:
TOTAL_GB = 2000
NODES = 8
GB_PER_NODE_MINUTE = 5.0
OVERHEAD = 1.25
COST_PER_NODE_HOUR = 6.0


def transform_cost(gb, nodes=NODES):
    runtime_min = gb / (GB_PER_NODE_MINUTE * nodes)
    billed_min = runtime_min * OVERHEAD
    node_hours = nodes * billed_min / 60
    return runtime_min, node_hours, node_hours * COST_PER_NODE_HOUR


full_min, full_hours, full_cost = transform_cost(TOTAL_GB)
inc_min, inc_hours, inc_cost = transform_cost(TOTAL_GB * 0.05)

print(f"full refresh: {full_min:.0f} min, {full_hours:.3f} node-hours, {full_cost:.1f}")
print(f"incremental : {inc_min:.1f} min, {inc_hours:.3f} node-hours, {inc_cost:.1f}")

assert full_min == 50.0
assert round(full_hours, 3) == 8.333
assert full_cost == 50.0
assert inc_cost == 2.5
assert round(full_cost / inc_cost) == 20

### Try it 9. Compare batch and streaming freshness and cost

**Task.** A batch job runs every 60 minutes and costs 18.00 per run. A streaming job runs every 5 minutes and costs 5.00 per run. Assume stale data is, on average, half the interval. Compute the daily cost and average freshness of each, then the cost of each minute of freshness removed.

**Hint.** Runs per day are `1440 / interval_minutes`. Average latency is `interval_minutes / 2`. Divide the cost difference by the freshness difference.

**Expected result.** Batch costs `432.0` per day at `30.0` minutes average freshness. Streaming costs `1440.0` per day at `2.5` minutes, so each minute of freshness costs about `36.65` dollars per day.

In [ ]:
BATCH_INTERVAL = 60
BATCH_COST_PER_RUN = 18.0
STREAM_INTERVAL = 5
STREAM_COST_PER_RUN = 5.0
MINUTES_PER_DAY = 24 * 60


def profile(interval, cost_per_run):
    runs = MINUTES_PER_DAY / interval
    return runs, runs * cost_per_run, interval / 2


b_runs, b_cost, b_fresh = profile(BATCH_INTERVAL, BATCH_COST_PER_RUN)
s_runs, s_cost, s_fresh = profile(STREAM_INTERVAL, STREAM_COST_PER_RUN)
dollars_per_minute = (s_cost - b_cost) / (b_fresh - s_fresh)

print(f"batch    : {b_runs:.0f} runs, {b_cost:.1f}/day, {b_fresh:.1f} min fresh")
print(f"streaming: {s_runs:.0f} runs, {s_cost:.1f}/day, {s_fresh:.1f} min fresh")
print(f"freshness costs {dollars_per_minute:.2f} per minute per day")

assert b_cost == 432.0 and b_fresh == 30.0
assert s_cost == 1440.0 and s_fresh == 2.5
assert round(dollars_per_minute, 2) == 36.65

### Try it 10. Plan a backfill window from throughput

**Task.** You must rebuild 365 days for a table that grows by 50,000,000 rows per day. The end-to-end rate, including transform and load, is 12,000 rows per second per stream. Compute the serial wall-clock time, then the time when 6 streams run in parallel.

**Hint.** `seconds = total_rows / effective_rows_per_second`. Divide by the stream count or by the number of hours in the nightly window to get the practical schedule.

**Expected result.** About `1,520,833` seconds, or `422.45` hours serial, falling to about `70.41` hours or `2.93` days with 6 streams.

In [ ]:
HISTORY_DAYS = 365
ROWS_PER_DAY = 50_000_000
ROWS_PER_SECOND = 12_000
STREAMS = 6

total_rows = HISTORY_DAYS * ROWS_PER_DAY
serial_seconds = total_rows / ROWS_PER_SECOND
serial_hours = serial_seconds / 3600
parallel_hours = serial_hours / STREAMS

print(f"total rows     = {total_rows:,}")
print(f"serial         = {serial_seconds:,.0f} s = {serial_hours:.2f} h")
print(f"with {STREAMS} streams  = {parallel_hours:.2f} h = {parallel_hours / 24:.2f} days")

assert total_rows == 18_250_000_000
assert round(serial_seconds) == 1_520_833
assert round(serial_hours, 2) == 422.45
assert round(parallel_hours, 2) == 70.41
assert round(parallel_hours / 24, 2) == 2.93

### Try it 11. Estimate concurrency needs from user counts and query time

**Task.** 400 analysts each run 6 queries per hour. At peak, activity triples. An average query occupies its slot for 20 seconds. Compute the query-seconds demanded per peak hour, the number of concurrently busy slots, and the number of 8-slot warehouses needed.

**Hint.** Demand in query-seconds is `users * queries_per_hour * peak_factor * seconds`. The slot count is that demand divided by 3600 seconds. Round warehouses up.

**Expected result.** `144,000` query-seconds per hour, `40.0` busy slots, and `5` warehouses of 8 slots.

In [ ]:
import math

USERS = 400
QUERIES_PER_USER_HOUR = 6
PEAK_FACTOR = 3
SECONDS_PER_QUERY = 20
SLOTS_PER_WAREHOUSE = 8

demand_seconds = USERS * QUERIES_PER_USER_HOUR * PEAK_FACTOR * SECONDS_PER_QUERY
busy_slots = demand_seconds / 3600
warehouses = math.ceil(busy_slots / SLOTS_PER_WAREHOUSE)

print(f"demand   = {demand_seconds:,} query-seconds per peak hour")
print(f"slots    = {busy_slots:.1f} concurrently busy")
print(f"warehouse= {warehouses} x {SLOTS_PER_WAREHOUSE} slots")

assert demand_seconds == 144_000
assert busy_slots == 40.0
assert warehouses == 5

### Try it 12. Compute a cost per report

**Task.** The warehouse costs 48,000 per month. Dashboard refresh consumes 20 percent of that. There are 1,500 reports, each refreshed once per day for 30 days. Compute the cost per refresh and the monthly cost per report.

**Hint.** Monthly refreshes are `reports * days`. Cost per refresh is the allocated spend divided by refreshes; cost per report is the allocated spend divided by reports.

**Expected result.** `45,000` refreshes, `0.213` per refresh, and `6.40` per report per month.

In [ ]:
MONTHLY_COST = 48_000.0
DASHBOARD_SHARE = 0.20
REPORTS = 1_500
DAYS = 30

dashboard_cost = MONTHLY_COST * DASHBOARD_SHARE
refreshes = REPORTS * DAYS
cost_per_refresh = dashboard_cost / refreshes
cost_per_report = dashboard_cost / REPORTS

print(f"dashboard spend  = {dashboard_cost:,.0f}/month")
print(f"refreshes        = {refreshes:,}")
print(f"cost per refresh = {cost_per_refresh:.3f}")
print(f"cost per report  = {cost_per_report:.2f}/month")

assert dashboard_cost == 9_600.0
assert refreshes == 45_000
assert round(cost_per_refresh, 3) == 0.213
assert round(cost_per_report, 2) == 6.40

### Try it 13. Detect an idempotency defect from a re-run scenario

**Task.** A pipeline appends one partition keyed by `order_id`. The loader produced 50,000 rows for the day. The task is retried after a transient failure and appends the same 50,000 rows again. Model the append, detect the duplicate keys, and show what the row count should be after a delete-then-insert re-run.

**Hint.** Build the partition as a list of keys. A second append doubles the list. The distinct count stays 50,000, so `len(partition) != len(set(partition))` is the defect signal.

**Expected result.** The partition holds `100,000` rows with `50,000` distinct keys, so the defect is `True`. The idempotent re-run leaves `50,000` rows and defect `False`.

In [ ]:
ROWS_IN_BATCH = 50_000

# Append-style load: the retry adds the same keys again.
partition = list(range(ROWS_IN_BATCH))
partition.extend(range(ROWS_IN_BATCH))

duplicates = len(partition) - len(set(partition))
defect = duplicates > 0
print(f"append rows    = {len(partition):,}")
print(f"distinct keys  = {len(set(partition)):,}")
print(f"duplicates     = {duplicates:,}, idempotency defect = {defect}")

assert len(partition) == 100_000
assert len(set(partition)) == 50_000
assert defect is True

# Idempotent load: delete the interval, then insert once.
partition = []
partition.extend(range(ROWS_IN_BATCH))
print(f"after idempotent re-run rows = {len(partition):,}")
assert len(partition) == 50_000
assert len(partition) == len(set(partition))
assert (len(partition) - len(set(partition))) == 0